In [1]:
import json
import os
import sys
import uuid
from pathlib import Path
from typing import Any

from tqdm.auto import tqdm

# ── Configuration ─────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\phili\PycharmProjects\UDA")

DOCLING_OUTPUT = PROJECT_ROOT / "data" / "src" / "docling_output"
CHUNKS_DIR = str(sorted(DOCLING_OUTPUT.iterdir())[-1])
print(f"Using chunks from: {CHUNKS_DIR}")

QDRANT_PATH = str(PROJECT_ROOT / "data" / "qdrant_db")
COLLECTION_NAME = "annual_reports"
BGE_M3_MODEL = "BAAI/bge-m3"
BATCH_SIZE_EMBED = 64                   # Adjust down if OOM
BATCH_SIZE_UPSERT = 100
RECREATE_COLLECTION = False             # Set True to wipe and recreate

UUID_NAMESPACE = uuid.NAMESPACE_URL
REQUIRED_FIELDS = {"chunk_id", "pdf_name", "type", "page", "text"}

Using chunks from: C:\Users\phili\PycharmProjects\UDA\data\src\docling_output\20260312_121205__kopie


C:\Users\phili\PycharmProjects\UDA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Load and validate chunks

In [2]:
def load_chunks_from_file(filepath: Path) -> list[dict[str, Any]]:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        data = data["chunks"]
    if not isinstance(data, list):
        raise ValueError(f"{filepath.name}: expected JSON array, got {type(data).__name__}")
    return data


def validate_chunk(chunk: dict[str, Any]) -> list[str]:
    """Validate a single chunk. Returns list of error messages (empty = valid)."""
    errors = []
    missing = REQUIRED_FIELDS - set(chunk.keys())
    if missing:
        errors.append(f"Missing fields: {missing}")
    if "type" in chunk and chunk["type"] not in ("text", "table"):
        errors.append(f"Invalid type: {chunk['type']!r}")
    if "page" in chunk and not isinstance(chunk["page"], int):
        errors.append(f"page should be int, got {type(chunk['page']).__name__}")
    return errors


def extract_ticker_year(pdf_name: str) -> tuple[str, int]:
    """Extract ticker and year from pdf_name like 'AAL_2010'."""
    parts = pdf_name.split("_")
    if len(parts) < 2:
        raise ValueError(f"Cannot parse pdf_name: {pdf_name!r}")
    return parts[0], int(parts[1])


def load_all_chunks(chunks_dir: str) -> list[dict[str, Any]]:
    """Load all *_chunks.json files from the directory."""
    chunks_path = Path(chunks_dir)
    if not chunks_path.exists():
        print(f"ERROR: Chunks directory not found: {chunks_dir}")
        return []

    json_files = sorted(chunks_path.glob("*_chunks.json"))
    if not json_files:
        print(f"ERROR: No *_chunks.json files found in {chunks_dir}")
        return []

    print(f"Found {len(json_files)} chunk file(s) in {chunks_dir}")

    all_chunks = []
    total_errors = 0

    for filepath in json_files:
        try:
            file_chunks = load_chunks_from_file(filepath)
        except (json.JSONDecodeError, ValueError) as e:
            print(f"  SKIP {filepath.name}: {e}")
            continue

        valid_count = 0
        for i, chunk in enumerate(file_chunks):
            errors = validate_chunk(chunk)
            if errors:
                print(f"  WARN {filepath.name}[{i}] chunk_id={chunk.get('chunk_id', '?')}: {errors}")
                total_errors += 1
                continue

            try:
                ticker, year = extract_ticker_year(chunk["pdf_name"])
                chunk["ticker"] = ticker
                chunk["year"] = year
            except ValueError as e:
                print(f"  WARN {filepath.name}[{i}]: {e}")
                total_errors += 1
                continue

            valid_count += 1
            all_chunks.append(chunk)

        print(f"  {filepath.name}: {valid_count}/{len(file_chunks)} chunks valid")

    print(f"\nTotal chunks loaded: {len(all_chunks)}  |  Errors: {total_errors}")

    if all_chunks:
        pdf_names = set(c["pdf_name"] for c in all_chunks)
        type_counts = {}
        for c in all_chunks:
            type_counts[c["type"]] = type_counts.get(c["type"], 0) + 1
        token_counts = [c.get("token_count", 0) for c in all_chunks]
        print(f"Documents: {len(pdf_names)}  |  Types: {type_counts}")
        print(f"Tokens: min={min(token_counts)}, max={max(token_counts)}, avg={sum(token_counts)/len(token_counts):.0f}")

    return all_chunks

In [3]:
chunks = load_all_chunks(CHUNKS_DIR)

# Show a few samples
for c in chunks[:3]:
    print(f"  {c['chunk_id']} | type={c['type']} | page={c['page']} | tokens={c.get('token_count','?')}")
    print(f"    text: {c['text'][:100]}...")
    if c.get("html"):
        print(f"    html: {c['html'][:80]}...")
    print()

Found 47 chunk file(s) in C:\Users\phili\PycharmProjects\UDA\data\src\docling_output\20260312_121205__kopie
  AAL_2010_chunks.json: 817/817 chunks valid
  AAL_2013_chunks.json: 1800/1800 chunks valid
  AAL_2014_chunks.json: 1913/1913 chunks valid
  AAL_2015_chunks.json: 1671/1671 chunks valid
  AAL_2016_chunks.json: 3054/3054 chunks valid
  AAL_2017_chunks.json: 1556/1556 chunks valid
  AAL_2018_chunks.json: 1614/1614 chunks valid
  AAP_2006_chunks.json: 587/587 chunks valid
  AAP_2007_chunks.json: 595/595 chunks valid
  AAP_2011_chunks.json: 1148/1148 chunks valid
  AAP_2012_chunks.json: 594/594 chunks valid
  AAP_2013_chunks.json: 615/615 chunks valid
  AAP_2016_chunks.json: 660/660 chunks valid
  AAPL_2002_chunks.json: 551/551 chunks valid
  AAPL_2003_chunks.json: 739/739 chunks valid
  AAPL_2004_chunks.json: 674/674 chunks valid
  AAPL_2005_chunks.json: 710/710 chunks valid
  AAPL_2006_chunks.json: 932/932 chunks valid
  AAPL_2007_chunks.json: 718/718 chunks valid
  AAPL_2008_chunk

## Step 2 — Initialize Qdrant collection

In [4]:
from qdrant_client import QdrantClient, models

def init_qdrant(qdrant_path: str, collection_name: str, recreate: bool = False) -> QdrantClient:
    """Initialize Qdrant client and create collection with schema."""
    client = QdrantClient(path=qdrant_path)
    print(f"Qdrant client initialized (path={qdrant_path})")

    existing = [c.name for c in client.get_collections().collections]

    if collection_name in existing:
        if recreate:
            print(f"Recreating collection '{collection_name}'...")
            client.delete_collection(collection_name)
        else:
            info = client.get_collection(collection_name)
            print(f"Collection '{collection_name}' exists — {info.points_count} points. Skipping creation.")
            return client

    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense": models.VectorParams(
                size=1024,
                distance=models.Distance.COSINE,
            ),
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                modifier=models.Modifier.IDF,
            ),
        },
    )
    print(f"Collection '{collection_name}' created (dense=1024d cosine, sparse=IDF)")

    # Payload indexes (effective in server mode, no-op in embedded mode)
    for field, schema_type in [
        ("pdf_name", models.PayloadSchemaType.KEYWORD),
        ("ticker",   models.PayloadSchemaType.KEYWORD),
        ("year",     models.PayloadSchemaType.INTEGER),
        ("type",     models.PayloadSchemaType.KEYWORD),
        ("section",  models.PayloadSchemaType.KEYWORD),
        ("page",     models.PayloadSchemaType.INTEGER),
    ]:
        client.create_payload_index(collection_name, field, schema_type)

    print("Payload indexes created (6 fields)")
    return client

In [5]:
client = init_qdrant(QDRANT_PATH, COLLECTION_NAME, recreate=RECREATE_COLLECTION)

Qdrant client initialized (path=C:\Users\phili\PycharmProjects\UDA\data\qdrant_db)
Collection 'annual_reports' created (dense=1024d cosine, sparse=IDF)
Payload indexes created (6 fields)


C:\Users\phili\AppData\Local\Temp\ipykernel_41728\4239096256.py:44: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(collection_name, field, schema_type)


## Step 3 & 4 — Embed with BGE-M3 and upsert to Qdrant

In [6]:
from FlagEmbedding import BGEM3FlagModel
from qdrant_client.models import PointStruct, SparseVector


def to_sparse_vector(lexical_weights: dict) -> SparseVector:
    """Convert BGE-M3 lexical_weights dict to Qdrant SparseVector."""
    if not lexical_weights:
        return SparseVector(indices=[], values=[])
    indices = list(lexical_weights.keys())
    values = list(lexical_weights.values())
    return SparseVector(
        indices=[int(i) for i in indices],
        values=[float(v) for v in values],
    )


def chunk_to_point_id(chunk_id: str) -> str:
    """Generate deterministic UUID5 from chunk_id."""
    return str(uuid.uuid5(UUID_NAMESPACE, chunk_id))


def build_payload(chunk: dict[str, Any]) -> dict[str, Any]:
    """Build Qdrant payload from chunk. Excludes markdown/serialized."""
    return {
        "chunk_id":    chunk["chunk_id"],
        "pdf_name":    chunk["pdf_name"],
        "ticker":      chunk["ticker"],
        "year":        chunk["year"],
        "type":        chunk["type"],
        "page":        chunk["page"],
        "section":     chunk.get("section"),
        "token_count": chunk.get("token_count"),
        "text":        chunk["text"],
        "html":        chunk.get("html"),
    }


def embed_and_upsert(
    client: QdrantClient,
    collection_name: str,
    chunks: list[dict[str, Any]],
    bge_model: BGEM3FlagModel,
    batch_size_embed: int = 64,
    batch_size_upsert: int = 100,
) -> None:
    """Generate embeddings and upsert all chunks into Qdrant."""
    print(f"Embedding and upserting {len(chunks)} chunks...")
    print(f"  Embed batch size:  {batch_size_embed}")
    print(f"  Upsert batch size: {batch_size_upsert}")

    points_buffer: list[PointStruct] = []
    upserted = 0

    for batch_start in tqdm(range(0, len(chunks), batch_size_embed), desc="Embedding"):
        batch = chunks[batch_start : batch_start + batch_size_embed]
        texts = [c["text"] for c in batch]

        output = bge_model.encode(
            texts,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
            batch_size=batch_size_embed,
        )

        for i, chunk in enumerate(batch):
            dense_vec = output["dense_vecs"][i].tolist()
            sparse_vec = to_sparse_vector(output["lexical_weights"][i])

            point = PointStruct(
                id=chunk_to_point_id(chunk["chunk_id"]),
                vector={
                    "dense": dense_vec,
                    "sparse": sparse_vec,
                },
                payload=build_payload(chunk),
            )
            points_buffer.append(point)

        # Flush upsert buffer
        while len(points_buffer) >= batch_size_upsert:
            batch_to_upsert = points_buffer[:batch_size_upsert]
            points_buffer = points_buffer[batch_size_upsert:]
            client.upsert(collection_name=collection_name, points=batch_to_upsert)
            upserted += len(batch_to_upsert)

    # Flush remaining
    if points_buffer:
        client.upsert(collection_name=collection_name, points=points_buffer)
        upserted += len(points_buffer)

    print(f"Upserted {upserted} points total.")

In [7]:
print(f"Loading BGE-M3 model ({BGE_M3_MODEL})...")
bge_model = BGEM3FlagModel(BGE_M3_MODEL, use_fp16=True, device="cuda")
print("Model loaded on GPU.")

Loading BGE-M3 model (BAAI/bge-m3)...


Fetching 30 files: 100%|██████████| 30/30 [00:00<?, ?it/s]


Model loaded on GPU.


In [8]:
embed_and_upsert(client, COLLECTION_NAME, chunks, bge_model, BATCH_SIZE_EMBED, BATCH_SIZE_UPSERT)

Embedding and upserting 37600 chunks...
  Embed batch size:  64
  Upsert batch size: 100


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 166.65it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 181.32it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 166.63it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 195.27it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 249.41it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 153.50it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 250.00it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 333.25it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 249.25it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 333.28it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 249.77it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 285.31it/s]

pre tokenize: 100%|█████

Upserted 37600 points total.


## Step 5 — Verification

In [9]:
info = client.get_collection(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}'")
print(f"  Points:         {info.points_count}")
print(f"  Status:         {info.status}")
print(f"  Dense vectors:  {info.config.params.vectors}")
print(f"  Sparse vectors: {info.config.params.sparse_vectors}")

Collection 'annual_reports'
  Points:         37600
  Status:         green
  Dense vectors:  {'dense': VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}
  Sparse vectors: {'sparse': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>)}


In [10]:
# Test query — unfiltered
test_question = "What were the fuel costs?"
query_output = bge_model.encode([test_question], return_dense=True, return_sparse=True, return_colbert_vecs=False)
query_dense = query_output["dense_vecs"][0].tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_dense,
    using="dense",
    limit=5,
)

print(f"Query: '{test_question}'\n")
for r in results.points:
    print(f"  score={r.score:.4f}  {r.payload['chunk_id']}  type={r.payload['type']}  page={r.payload['page']}")
    print(f"    {r.payload['text'][:120]}...")
    print()

Query: 'What were the fuel costs?'

  score=0.5958  AAL_2010_text_36  type=text  page=8
    The Company's operations and financial results are significantly affected by the availability and price of jet  fuel. Th...

  score=0.5885  AAL_2018_table_3  type=table  page=13
    Here's the data extracted from the provided HTML table, presented as plain text:

In 2018, there were 4,447 gallons with...

  score=0.5864  AAL_2010_text_133  type=text  page=29
    The increase in total passenger revenue was partially offset by  significantly higher year-over-year fuel prices. Fuel e...

  score=0.5823  AAL_2018_table_57  type=table  page=114
    Aircraft fuel and related taxes were $1,843 for the year ended December 31, 2018; $1,382 for the year ended December 31,...

  score=0.5805  AAL_2013_text_277  type=text  page=58
    Mainline and regional fuel expense was relatively flat at  $8.7 billion in each of 2013 and 2012. The average mainline a...



In [11]:
# Test query — filtered by document
results_filtered = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_dense,
    using="dense",
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="pdf_name",
                match=models.MatchValue(value="AAL_2010"),
            )
        ]
    ),
    limit=5,
)

print("Filtered query (pdf_name=AAL_2010):\n")
for r in results_filtered.points:
    print(f"  score={r.score:.4f}  {r.payload['chunk_id']}")

Filtered query (pdf_name=AAL_2010):

  score=0.5958  AAL_2010_text_36
  score=0.5864  AAL_2010_text_133
  score=0.5759  AAL_2010_text_38
  score=0.5741  AAL_2010_text_170
  score=0.5711  AAL_2010_table_0


In [12]:
# Test query — Hybrid (Dense + Sparse) with RRF fusion
from qdrant_client.models import Prefetch, FusionQuery, Fusion, SparseVector

test_question = "What were the fuel costs?"
query_output = bge_model.encode([test_question], return_dense=True, return_sparse=True, return_colbert_vecs=False)

query_dense = query_output["dense_vecs"][0].tolist()
query_sparse = to_sparse_vector(query_output["lexical_weights"][0])

results_hybrid = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        Prefetch(query=query_dense, using="dense", limit=20),
        Prefetch(query=query_sparse, using="sparse", limit=20),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=5,
)

print(f"Query (HYBRID): '{test_question}'\n")
for r in results_hybrid.points:
    print(f"  score={r.score:.4f}  {r.payload['chunk_id']}  type={r.payload['type']}  page={r.payload['page']}")
    print(f"    {r.payload['text'][:120]}...")
    print()

Query (HYBRID): 'What were the fuel costs?'

  score=1.0000  AAL_2010_text_36  type=text  page=8
    The Company's operations and financial results are significantly affected by the availability and price of jet  fuel. Th...

  score=0.3333  AAL_2018_table_3  type=table  page=13
    Here's the data extracted from the provided HTML table, presented as plain text:

In 2018, there were 4,447 gallons with...

  score=0.3333  AAL_2016_text_397  type=text  page=82
    Our operating results are materially impacted by changes in the availability, price volatility and cost of aircraft fuel...

  score=0.2500  AAL_2010_text_133  type=text  page=29
    The increase in total passenger revenue was partially offset by  significantly higher year-over-year fuel prices. Fuel e...

  score=0.2500  AAL_2015_text_517  type=text  page=100
    Our operating results are materially impacted by changes in the availability, price volatility and cost of aircraft fuel...



In [13]:
# Cleanup — close Qdrant client when done
# client.close()